# Simplified Copy-Trade Backtest

Simulates copying trades of a hand-picked set of wallets against the
processed Polygon trade shards.

## Strategy
- When a watched wallet prints a **BUY** on token `(condition_id, outcome)` at price `p`
  we place a buy order at a **slightly worse** price: `order_price = clip(p + WORSE_PRICE_OFFSET, 0.001, 0.999)`.
- The order is matched against the **first subsequent trade** whose effective price is
  `<= order_price` within `FILL_HORIZON_SECONDS`.
  - *same-token* liquidity: later BUY trades on the same `(condition_id, outcome)`.
  - *opposite-token* liquidity: later SELL trades on the **complementary** outcome
    (binary markets only), with effective price = `1 - sell_price`.
- For a wallet **SELL** the mirror applies: order at `p - WORSE_PRICE_OFFSET`, filled
  by the first later trade with effective price `>= order_price`.
  - same-token: later SELL prints on the same outcome.
  - opposite-token: later BUY prints on the complementary outcome, effective price `1 - buy_price`.

## Sharding
Trades are partitioned by the first hex character of `condition_id` (after `0x`).
All shards are processed in parallel; within each shard the backtest is exact
because a `condition_id` always falls in exactly one shard file.

## Outputs
One event ledger DataFrame with columns:
- `row_type` — `trigger` (watched-wallet trade that generated an order) or `fill` (our fill).
- `order_id` — unique integer identifying the copy order.
- `wallet` — originating wallet (`fill` rows carry the trigger wallet for reference).
- `condition_id`, `outcome`, `side`, `dt`, `tx_hash`, `price`.
- `order_price` — the price our order was placed at (`NaN` for fill rows).
- `fill_source` — `same_token` / `opposite_token` / `NaN` for trigger rows.
- `token_winner` — market resolution flag.
- `fill_pnl_usdc` — PnL of *our* position on fill rows (NaN for trigger rows),
  computed as stake × ( 1/exec_price − 1 ) for winning tokens, −stake for losers.
- `filled_wallet_total_position` — total filled position for this wallet across all conditions/outcomes.
- `filled_token_total_position` — total filled position for this `(condition_id, outcome)` across all wallets.
- `filled_token_by_wallet_position` — filled position for this `(wallet, condition_id, outcome)`.

## Visualisation
Cumulative PnL chart with two series:
- **Wallet cohort** — raw `trade_pnl` of the watched wallets.
- **Copy strategy** — `fill_pnl_usdc` of our simulated fills.


## Configuration

In [43]:
%load_ext autoreload
%autoreload 2

import datetime
from pathlib import Path

# ── Input wallets ─────────────────────────────────────────────────────────────
# Replace with any wallet addresses you want to copy.
# Example: load from a CSV, a workspace registry, or define inline.
WATCHED_WALLETS: list[str] = None

ONLY_BUY_TRIGGERS = False
MAX_EXPOSURE_PER_WALLET_1h = 200

# ── Test period ───────────────────────────────────────────────────────────────
# None = use entire dataset (start/end are inferred from the data).
# Set to datetime.date objects to restrict the window.
PERIOD_START: datetime.date | None = datetime.date(2026, 4, 16)
# PERIOD_START: datetime.date | None = datetime.date(2026, 1, 1) # None   # e.g. datetime.date(2026, 2, 16)
PERIOD_END:   datetime.date | None = datetime.date(2026, 6, 20)
# PERIOD_END:   datetime.date | None = datetime.date(2026, 2, 16) # None   # e.g. datetime.date(2026, 3, 11)

# ── Copy-trade execution parameters ──────────────────────────────────────────
WORSE_PRICE_OFFSET: float = 0
FILL_HORIZON_SECONDS: float = 300     # max seconds after trigger to search for a fill
STAKE_USDC: float = 10.0               # max USDC per order (order qty = min(trigger_qty, STAKE_USDC / order_price))
FEE_BPS: float = 10.0                   # taker fee in basis points
MAX_POSITION_PER_CONDITION: float | None = 1000  # max qty per (wallet, condition_id) across all outcomes; None = unlimited
MAX_POSITION_PER_WALLET:    float | None = 100  # max qty per wallet across all conditions; None = unlimited

# ── Data ─────────────────────────────────────────────────────────────────────
TRADES_DIR     = Path("../../data/polygon_trades_processed")
RAW_TRADES_DIR = Path("../../data/trades_polygon")

# ── Parallelism ───────────────────────────────────────────────────────────────
MAX_WORKERS: int = 10   # number of threads for parallel shard processing

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [44]:
banned_wallets = set()
if(WATCHED_WALLETS is not None):
    WATCHED_WALLETS = [w for w in WATCHED_WALLETS if w not in banned_wallets]

## Optionally load wallets from stage-1 workspace

If `WATCHED_WALLETS` is empty above, this cell loads the wallet set from the
stage-1 strategy registry. Skip or modify as needed.

In [45]:
import pandas as pd

if not WATCHED_WALLETS:
    WORKSPACE_DIR = Path("../../data/trade_signals_workspace_v2")
    wallets_path = WORKSPACE_DIR / "selected_wallets_v2.parquet"
    if wallets_path.exists():
        _wallets_df = pd.read_parquet(wallets_path, columns=["wallet"])
        WATCHED_WALLETS = _wallets_df["wallet"].tolist()
        print(f"Loaded {len(WATCHED_WALLETS)} wallets from {wallets_path.name}")
    else:
        print("No wallet source found — set WATCHED_WALLETS manually or run stage1 first.")
else:
    print(f"Using {len(WATCHED_WALLETS)} manually specified wallets.")

Loaded 22 wallets from selected_wallets_v2.parquet


## Discover shards and derive test period

In [46]:
import pandas as pd

SHARD_PATHS: list[Path] = sorted(TRADES_DIR.glob("*.parquet"))
print(f"Found {len(SHARD_PATHS)} shards under {TRADES_DIR}")

# Derive END_DATE_TRAIN from the is_train flag (last date where is_train=True).
# Test data is everything strictly after END_DATE_TRAIN.
_sample = pd.read_parquet(SHARD_PATHS[0], columns=["dt", "is_train"])
_sample["dt"] = pd.to_datetime(_sample["dt"], utc=True)
END_DATE_TRAIN: datetime.date = _sample.loc[_sample["is_train"], "dt"].max().date()
_data_end: datetime.date      = _sample["dt"].max().date()
del _sample
print(f"END_DATE_TRAIN (last train date) : {END_DATE_TRAIN}")

# Backtest always runs on test data only (strictly after END_DATE_TRAIN).
# PERIOD_START / PERIOD_END can narrow the window further, but cannot go
# earlier than the day after END_DATE_TRAIN.
#_test_start = END_DATE_TRAIN + datetime.timedelta(days=1)
period_start: datetime.date = PERIOD_START # datetime.date(2026, 2, 16) #  max(PERIOD_START, _test_start) if PERIOD_START is not None else _test_start
period_end:   datetime.date = PERIOD_END #if PERIOD_END is not None else _data_end
print(f"Backtest period : {period_start}  →  {period_end}")

Found 16 shards under ../../data/polygon_trades_processed
END_DATE_TRAIN (last train date) : 2026-04-15
Backtest period : 2026-04-16  →  2026-06-20


## Backtest engine (per-shard)

Each shard is processed independently:
1. Load only rows within the backtest period (test data only).
2. Identify trigger events (watched-wallet trades).
3. Build a per-`condition_id` opposite-outcome map (binary markets only).
4. Process triggers in time order, maintaining a copy-portfolio **position** per `(wallet, condition_id, outcome)`.
5. For each trigger, place a copy order:
   - **BUY**: `qty_ordered = min(trig_qty, STAKE_USDC / order_price)` — worst-case loss = `qty × order_price ≤ STAKE_USDC`
   - **SELL**: `qty_ordered = min(trig_qty, position, STAKE_USDC / (1 − order_price))` — worst-case loss = `qty × (1 − order_price) ≤ STAKE_USDC`. Skipped if position = 0 (no short selling).
6. Scan tape prints in time order within `FILL_HORIZON_SECONDS`. Each matching print partially fills the order: `fill_qty = min(remaining_qty, tape_qty)`.
7. One **fill** ledger row is emitted per partial fill. BUY fills increment position; SELL fills decrement it.

`fill_pnl_usdc` per fill row (all executed at `exec_price = order_price`, limit fill):
- **BUY winner**: `fill_qty × (1 − exec_price) − fill_qty × exec_price × fee`
- **BUY loser**:  `−fill_qty × exec_price × (1 + fee)`
- **SELL winner** (sold below par): `fill_qty × (exec_price − 1) − fill_qty × exec_price × fee`
- **SELL loser** (sold above par):  `fill_qty × exec_price × (1 − fee)`


In [47]:
import numpy as np
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

# ─────────────────────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────────────────────

def _build_complement_map(df: pd.DataFrame) -> dict[tuple[str, str], str]:
    """Return {(condition_id, outcome): opposite_outcome} for binary markets."""
    pairs: dict[str, set] = {}
    for cid, grp in df.groupby("condition_id", sort=False):
        pairs[cid] = set(grp["outcome"].dropna().unique())
    result: dict[tuple[str, str], str] = {}
    for cid, outcomes in pairs.items():
        if len(outcomes) == 2:
            a, b = sorted(outcomes)
            result[(cid, a)] = b
            result[(cid, b)] = a
    return result


def _simulate_shard(
    shard_path: Path,
    raw_tape_path: Path,
    watched_wallets: set[str],
    period_start: datetime.date,
    period_end: datetime.date,
    worse_price_offset: float,
    fill_horizon_seconds: float,
    stake_usdc: float,
    fee_bps: float,
    max_position_per_condition: float | None = None,
    max_position_per_wallet: float | None = None,
) -> pd.DataFrame:
    """Process one shard file and return an event ledger DataFrame.

    Triggers are read from ``shard_path`` (processed per-wallet aggregated trades),
    which supplies ``token_winner`` and ``trade_pnl``.

    The fill tape is read from ``raw_tape_path`` (raw on-chain individual fills).
    Orders are simulated chronologically against the tape so that each tape print's
    quantity can only be consumed once across all active copy orders in the shard.
    """
    TRIG_COLS = [
        "wallet", "condition_id", "outcome", "dt", "side",
        "avg_price", "total_quantity", "trade_pnl", "token_winner",
        "tx_hash",
    ]
    tdf = pd.read_parquet(shard_path, columns=TRIG_COLS)
    if tdf.empty:
        return pd.DataFrame()

    tdf["dt"] = pd.to_datetime(tdf["dt"], utc=True)
    tdf = tdf.rename(columns={"avg_price": "price", "total_quantity": "quantity"})

    date_mask = (
        (tdf["dt"].dt.date >= period_start)
        & (tdf["dt"].dt.date <= period_end)
    )
    tdf = tdf[date_mask].copy()
    if tdf.empty:
        return pd.DataFrame()

    tdf["price"] = tdf["price"].astype(float)
    tdf["quantity"] = tdf["quantity"].astype(float)

    if ONLY_BUY_TRIGGERS:
        trigger_mask = (tdf["wallet"].isin(watched_wallets)) & (tdf["side"] == "BUY")
    else:
        trigger_mask = tdf["wallet"].isin(watched_wallets)
    triggers_df = tdf[trigger_mask].copy()
    if triggers_df.empty:
        return pd.DataFrame()

    tw_map: dict[tuple[str, str], bool] = {}
    for row in tdf[["condition_id", "outcome", "token_winner"]].itertuples(index=False):
        key = (row.condition_id, row.outcome)
        if key not in tw_map and row.token_winner is not None:
            tw_map[key] = bool(row.token_winner)

    complement = _build_complement_map(tdf)
    triggers_df = triggers_df.sort_values("dt", kind="mergesort").reset_index(drop=True)

    TAPE_COLS = ["condition_id", "outcome", "block_timestamp", "side", "price", "quantity", "tx_hash", "token_winner"]
    if raw_tape_path.exists():
        rdf = pd.read_parquet(raw_tape_path, columns=TAPE_COLS)
    else:
        rdf = pd.DataFrame(columns=TAPE_COLS)

    if rdf.empty:
        ledger_rows = []
        for trig in triggers_df.itertuples(index=False):
            cid = trig.condition_id
            outcome = trig.outcome
            side = trig.side
            trig_dt = trig.dt
            trig_px = float(trig.price)
            trig_qty = float(trig.quantity)
            trig_tw = bool(trig.token_winner)
            wallet = trig.wallet
            if side == "BUY":
                order_px = float(np.clip(trig_px + worse_price_offset, 0.001, 0.999))
                qty_ordered = min(trig_qty, stake_usdc / order_px)
            else:
                order_px = float(np.clip(trig_px - worse_price_offset, 0.001, 0.999))
                qty_ordered = trig_qty
            ledger_rows.append({
                "row_type": "trigger", "order_id": len(ledger_rows) + 1,
                "wallet": wallet, "condition_id": cid, "outcome": outcome,
                "side": side, "dt": trig_dt, "tx_hash": trig.tx_hash,
                "price": trig_px, "order_price": order_px,
                "qty_ordered": qty_ordered, "qty_filled": 0.0,
                "fill_qty": None, "fill_source": None,
                "token_winner": trig_tw, "exec_price": None,
                "fill_pnl_usdc": None,
                "wallet_trade_pnl": float(trig.trade_pnl) if trig.trade_pnl is not None else None,
                "filled_wallet_total_position": 0.0,
                "filled_token_total_position": 0.0,
                "filled_token_by_wallet_position": 0.0,
            })
        result = pd.DataFrame(ledger_rows)
        result["shard"] = shard_path.stem
        return result

    rdf["dt"] = pd.to_datetime(rdf["block_timestamp"], unit="s", utc=True)
    rdf = rdf.drop(columns=["block_timestamp"])

    tape_start = pd.Timestamp(period_start, tz="UTC")
    tape_end = pd.Timestamp(period_end, tz="UTC") + pd.Timedelta(days=1)
    rdf = rdf[(rdf["dt"] >= tape_start) & (rdf["dt"] < tape_end)].copy()
    rdf["price"] = rdf["price"].astype(float)
    rdf["quantity"] = rdf["quantity"].astype(float)
    rdf = rdf.sort_values("dt", kind="mergesort").reset_index(drop=True)

    fee = fee_bps / 10_000.0
    horizon = pd.Timedelta(seconds=fill_horizon_seconds)
    eps = 1e-12

    ledger_rows: list[dict] = []
    order_counter = 0
    # (wallet, condition_id, outcome) → filled qty for this outcome
    positions: dict[tuple[str, str, str], float] = defaultdict(float)
    # (wallet, condition_id) → total filled qty across all outcomes of that condition
    cond_positions: dict[tuple[str, str], float] = defaultdict(float)
    # (condition_id, outcome) → total filled qty for this outcome across all wallets
    token_total_positions: dict[tuple[str, str], float] = defaultdict(float)
    # wallet → total filled qty across all conditions
    wallet_positions: dict[str, float] = defaultdict(float)

    orders: dict[int, dict] = {}
    books: dict[tuple[str, str, str], list[dict]] = defaultdict(list)

    def _append_trigger_row(
        order_id: int,
        trig,
        order_px: float,
        qty_ordered: float,
        trig_tw: bool,
        current_token_pos: float = 0.0,
        current_token_total_pos: float = 0.0,
        current_wallet_pos: float = 0.0,
    ) -> None:
        ledger_rows.append({
            "row_type": "trigger",
            "order_id": order_id,
            "wallet": trig.wallet,
            "condition_id": trig.condition_id,
            "outcome": trig.outcome,
            "side": trig.side,
            "dt": trig.dt,
            "tx_hash": trig.tx_hash,
            "price": float(trig.price),
            "order_price": order_px,
            "qty_ordered": qty_ordered,
            "qty_filled": None,
            "fill_qty": None,
            "fill_source": None,
            "token_winner": trig_tw,
            "exec_price": None,
            "fill_pnl_usdc": None,
            "wallet_trade_pnl": float(trig.trade_pnl) if trig.trade_pnl is not None else None,
            "filled_wallet_total_position": current_wallet_pos,
            "filled_token_total_position": current_token_total_pos,
            "filled_token_by_wallet_position": current_token_pos,
        })

    # order can match with trade on the same side and token, or opposite side and opposite token    
    def _register_order(order_id: int, order: dict, trig_tw: bool) -> None:
        cid = order["condition_id"]
        outcome = order["outcome"]
        side = order["side"]
        books[(cid, outcome, side)].append({
            "order_id": order_id,
            "fill_source": "same_token",
            "fill_tw": trig_tw,
            "opposite": False,
        })
        opp_outcome = complement.get((cid, outcome))
        if opp_outcome is None:
            return
        opp_tw = tw_map.get((cid, opp_outcome), not trig_tw)
        opp_tape_side = "SELL" if side == "BUY" else "BUY"
        books[(cid, opp_outcome, opp_tape_side)].append({
            "order_id": order_id,
            "fill_source": "opposite_token",
            "fill_tw": trig_tw,
            "opposite": True,
        })

    def _process_tape_row(row) -> None:
        key = (row.condition_id, row.outcome, row.side)
        entries = books.get(key)
        if not entries:
            return

        tape_dt = row.dt
        tape_price = float(row.price)
        tape_remaining = float(row.quantity)
        if tape_remaining <= eps:
            return

        survivors: list[dict] = []
        for entry in entries:
            order = orders.get(entry["order_id"])
            if order is None:
                continue
            if order["remaining_qty"] <= eps:
                continue
            if order["deadline"] < tape_dt:
                continue

            eff_price = float(np.clip(1.0 - tape_price, 0.001, 0.999)) if entry["opposite"] else tape_price
            eligible = eff_price <= order["order_price"] if order["side"] == "BUY" else eff_price >= order["order_price"]

            if eligible and tape_remaining > eps:
                fill_qty = min(order["remaining_qty"], tape_remaining)

                # Cap fill to remaining position headroom (per-condition and per-wallet limits).
                # This prevents overfill when a single tape print is large enough to exceed the cap.
                if order["side"] == "BUY":
                    if max_position_per_condition is not None:
                        cond_headroom = max(max_position_per_condition - cond_positions[(order["wallet"], order["condition_id"])], 0.0)
                        fill_qty = min(fill_qty, cond_headroom)
                    if max_position_per_wallet is not None:
                        wallet_headroom = max(max_position_per_wallet - wallet_positions[order["wallet"]], 0.0)
                        fill_qty = min(fill_qty, wallet_headroom)
                    if fill_qty <= eps:
                        # Position cap already reached — retire this order without consuming tape
                        order["remaining_qty"] = 0.0
                        continue

                tape_remaining -= fill_qty
                order["remaining_qty"] -= fill_qty

                if order["side"] == "BUY":
                    gross = fill_qty * (1.0 - order["order_price"]) if entry["fill_tw"] else -fill_qty * order["order_price"]
                    fill_pnl = gross - fill_qty * order["order_price"] * fee
                    positions[order["pos_key"]] += fill_qty
                    cond_positions[(order["wallet"], order["condition_id"])] += fill_qty
                    token_total_positions[(order["condition_id"], order["outcome"])] += fill_qty
                    wallet_positions[order["wallet"]] += fill_qty
                    new_token_pos = positions[order["pos_key"]]
                    new_token_total_pos = token_total_positions[(order["condition_id"], order["outcome"])]
                    new_wallet_pos = wallet_positions[order["wallet"]]
                else:
                    gross = fill_qty * (order["order_price"] - 1.0) if entry["fill_tw"] else fill_qty * order["order_price"]
                    fill_pnl = gross - fill_qty * order["order_price"] * fee
                    prev = positions[order["pos_key"]]
                    positions[order["pos_key"]] = max(prev - fill_qty, 0.0)
                    actually_reduced = prev - positions[order["pos_key"]]
                    cond_positions[(order["wallet"], order["condition_id"])] = max(
                        cond_positions[(order["wallet"], order["condition_id"])] - actually_reduced, 0.0
                    )
                    wallet_positions[order["wallet"]] = max(
                        wallet_positions[order["wallet"]] - actually_reduced, 0.0
                    )
                    token_total_positions[(order["condition_id"], order["outcome"])] = max(
                        token_total_positions[(order["condition_id"], order["outcome"])] - actually_reduced, 0.0
                    )
                    new_token_pos = positions[order["pos_key"]]
                    new_token_total_pos = token_total_positions[(order["condition_id"], order["outcome"])]
                    new_wallet_pos = wallet_positions[order["wallet"]]

                ledger_rows.append({
                    "row_type": "fill",
                    "order_id": entry["order_id"],
                    "wallet": order["wallet"],
                    "condition_id": order["condition_id"],
                    "outcome": order["outcome"],
                    "side": order["side"],
                    "dt": tape_dt,
                    "tx_hash": row.tx_hash,
                    "price": eff_price,
                    "order_price": None,
                    "qty_ordered": order["qty_ordered"],
                    "qty_filled": None,
                    "fill_qty": fill_qty,
                    "fill_source": entry["fill_source"],
                    "token_winner": entry["fill_tw"],
                    "exec_price": order["order_price"],
                    "fill_pnl_usdc": fill_pnl,
                    "wallet_trade_pnl": None,
                    "filled_wallet_total_position": new_wallet_pos,
                    "filled_token_total_position": new_token_total_pos,
                    "filled_token_by_wallet_position": new_token_pos,
                })

            if order["remaining_qty"] > eps and order["deadline"] >= tape_dt:
                survivors.append(entry)

        if survivors:
            books[key] = survivors
        else:
            books.pop(key, None)

    tape_iter = iter(rdf[["condition_id", "outcome", "dt", "side", "price", "quantity", "tx_hash", "token_winner"]].itertuples(index=False))
    current_tape = next(tape_iter, None)

    for trig in triggers_df.itertuples(index=False):
        while current_tape is not None and current_tape.dt <= trig.dt:
            _process_tape_row(current_tape)
            current_tape = next(tape_iter, None)

        cid = trig.condition_id
        outcome = trig.outcome
        side = trig.side
        trig_px = float(trig.price)
        trig_qty = float(trig.quantity)
        trig_tw = bool(trig.token_winner)
        wallet = trig.wallet
        pos_key = (wallet, cid, outcome)

        if side == "BUY":
            order_px = float(np.clip(trig_px + worse_price_offset, 0.001, 0.999))
            current_pos = positions.get(pos_key, 0.0)
            qty_ordered = min(trig_qty, stake_usdc / order_px)
        else:
            order_px = float(np.clip(trig_px - worse_price_offset, 0.001, 0.999))
            current_pos = positions.get(pos_key, 0.0)
            if current_pos <= eps:
                continue
            sell_cap = stake_usdc / max(1.0 - order_px, 1e-9)
            qty_ordered = min(trig_qty, current_pos, sell_cap)
            if qty_ordered <= eps:
                continue

        order_counter += 1
        order_id = order_counter

        order = {
            "wallet": wallet,
            "condition_id": cid,
            "outcome": outcome,
            "side": side,
            "order_price": order_px,
            "qty_ordered": qty_ordered,
            "remaining_qty": qty_ordered,
            "deadline": trig.dt + horizon,
            "pos_key": pos_key,
        }
        orders[order_id] = order

        _append_trigger_row(
            order_id,
            trig,
            order_px,
            qty_ordered,
            trig_tw,
            current_token_pos=positions.get(pos_key, 0.0),
            current_token_total_pos=token_total_positions.get((cid, outcome), 0.0),
            current_wallet_pos=wallet_positions.get(wallet, 0.0),
        )
        _register_order(order_id, order, trig_tw)

    while current_tape is not None:
        _process_tape_row(current_tape)
        current_tape = next(tape_iter, None)

    if not ledger_rows:
        return pd.DataFrame()

    result = pd.DataFrame(ledger_rows)
    result["shard"] = shard_path.stem

    filled_by_order = (
        result[result["row_type"] == "fill"]
        .groupby("order_id")["fill_qty"]
        .sum()
        .rename("_total_filled")
    )
    result = result.merge(filled_by_order, on="order_id", how="left")
    trig_mask = result["row_type"] == "trigger"
    result.loc[trig_mask, "qty_filled"] = result.loc[trig_mask, "_total_filled"].fillna(0.0)
    result = result.drop(columns=["_total_filled"])
    result["qty_filled"] = result["qty_filled"].astype(float)

    return result



## Run backtest across all shards (parallel)

In [48]:
assert WATCHED_WALLETS, "WATCHED_WALLETS is empty — set wallets in the config cell or run the wallet-load cell."

watched_set = set(WATCHED_WALLETS)
shard_results: list[pd.DataFrame] = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(
            _simulate_shard,
            path,
            RAW_TRADES_DIR / path.name,
            watched_set,
            period_start,
            period_end,
            WORSE_PRICE_OFFSET,
            FILL_HORIZON_SECONDS,
            STAKE_USDC,
            FEE_BPS,
            MAX_POSITION_PER_CONDITION,
            MAX_POSITION_PER_WALLET,
        ): path
        for path in SHARD_PATHS
    }
    for future in as_completed(futures):
        path = futures[future]
        try:
            df = future.result()
            if df is not None and not df.empty:
                shard_results.append(df)
        except Exception as exc:
            print(f"Shard {path.name} raised an exception: {exc}")
            raise

if shard_results:
    event_ledger: pd.DataFrame = (
        pd.concat(shard_results, ignore_index=True)
        .sort_values(["dt", "shard", "order_id", "row_type"])
        .reset_index(drop=True)
    )
    _key = event_ledger[["shard", "order_id"]]
    _pairs = _key.drop_duplicates().reset_index(drop=True)
    _pairs["global_order_id"] = _pairs.index + 1
    event_ledger = event_ledger.merge(_pairs, on=["shard", "order_id"], how="left")
    event_ledger["order_id"] = event_ledger["global_order_id"]
    event_ledger = event_ledger.drop(columns=["global_order_id"])
else:
    event_ledger = pd.DataFrame(columns=[
        "row_type", "order_id", "wallet", "condition_id", "outcome",
        "side", "dt", "tx_hash", "price", "order_price",
        "qty_ordered", "qty_filled", "fill_qty",
        "fill_source", "token_winner", "exec_price", "fill_pnl_usdc",
        "wallet_trade_pnl", "shard", "filled_wallet_total_position", "filled_token_total_position", "filled_token_by_wallet_position",
    ])

triggers = event_ledger[event_ledger["row_type"] == "trigger"]
fills = event_ledger[event_ledger["row_type"] == "fill"]
filled_orders = fills["order_id"].nunique()

print(f"Trigger events    : {len(triggers):>7,}")
print(f"Fill rows         : {len(fills):>7,}")
print(f"Orders with fills : {filled_orders:>7,}")
if len(triggers) > 0:
    print(f"Order fill rate   : {filled_orders/len(triggers)*100:.1f}%")
else:
    print("No trigger events found — check WATCHED_WALLETS and the period.")



Trigger events    :   5,546
Fill rows         :   2,168
Orders with fills :   1,373
Order fill rate   : 24.8%


## Summary statistics

In [49]:
fee = FEE_BPS / 10_000.0

if not fills.empty:
    filled_copy_pnl    = fills["fill_pnl_usdc"].sum()
    total_qty_filled   = fills["fill_qty"].sum()
    total_notional     = (fills["fill_qty"] * fills["exec_price"]).sum()
    orders_with_fills  = fills["order_id"].nunique()
    partial_orders     = (fills.groupby("order_id").size() > 1).sum()
    win_rate           = fills.groupby("order_id")["token_winner"].first().mean()
    avg_exec_price     = fills["exec_price"].mean()
    fill_src_counts    = fills["fill_source"].value_counts()

    # Partial-fill statistics
    trig_qty = triggers.set_index("order_id")["qty_ordered"]
    fill_qty = triggers.set_index("order_id")["qty_filled"]
    fill_pct = (fill_qty / trig_qty.clip(lower=1e-12) * 100).replace([float('inf'), float('nan')], 0)

    print(f"=== Copy-strategy performance ===")
    print(f"  Orders triggered    : {len(triggers):>7,}")
    print(f"  Orders with fills   : {orders_with_fills:>7,}  ({orders_with_fills/len(triggers)*100:.1f}%)")
    print(f"  Orders multi-fill   : {partial_orders:>7,}  ({partial_orders/max(orders_with_fills,1)*100:.1f}% of filled)")
    print(f"  Avg fill %          : {fill_pct[fill_pct>0].mean():>7.1f}%")
    print(f"  Total qty filled    : {total_qty_filled:>10,.2f} tokens")
    print(f"  Total notional      : {total_notional:>10,.2f} USDC")
    print(f"  Net PnL (USDC)      : {filled_copy_pnl:>10,.2f}")
    print(f"  Net ROI on notional : {filled_copy_pnl/total_notional*100:>8.2f}%")
    print(f"  Win rate (by order) : {win_rate*100:>8.2f}%")
    print(f"  Avg exec price      : {avg_exec_price:>8.4f}")
    print(f"\n  Fill source breakdown:")
    for src, cnt in fill_src_counts.items():
        print(f"    {src:<20s}: {cnt:>6,}  ({cnt/len(fills)*100:.1f}%)")
else:
    print("No fills — nothing to summarise.")

# Watched-wallet cohort summary
wallet_pnl = triggers["wallet_trade_pnl"].sum()
print(f"\n=== Watched-wallet cohort ===")
print(f"  Total trades        : {len(triggers):>7,}")
print(f"  Total trade PnL     : {wallet_pnl:>10,.2f} USDC")


=== Copy-strategy performance ===
  Orders triggered    :   5,546
  Orders with fills   :   1,373  (24.8%)
  Orders multi-fill   :     462  (33.6% of filled)
  Avg fill %          :    88.4%
  Total qty filled    :  18,695.95 tokens
  Total notional      :   9,442.99 USDC
  Net PnL (USDC)      :     546.07
  Net ROI on notional :     5.78%
  Win rate (by order) :    57.61%
  Avg exec price      :   0.5701

  Fill source breakdown:
    same_token          :  1,554  (71.7%)
    opposite_token      :    614  (28.3%)

=== Watched-wallet cohort ===
  Total trades        :   5,546
  Total trade PnL     :  27,984.01 USDC


## Event ledger preview

In [50]:
event_ledger[
    (event_ledger['row_type'] == 'fill')
    & (event_ledger['condition_id'] == '0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86e6953305a2ef19c77f20')
    & (event_ledger['wallet'] == '0x0b219cf3d297991b58361dbebdbaa91e56b8deb6')
    ].head(500)

,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,...,fill_qty,fill_source,token_winner,exec_price,fill_pnl_usdc,wallet_trade_pnl,filled_wallet_total_position,filled_token_total_position,filled_token_by_wallet_position,shard


In [51]:
pd.set_option('display.max_rows', 100)

In [52]:
display_cols = [
    "row_type", "order_id", "wallet", "condition_id", "outcome", "side",
    "dt", "tx_hash", "price", "order_price", "exec_price",
    "qty_ordered", "qty_filled", "fill_qty",
    "fill_source", "token_winner", "fill_pnl_usdc", "filled_wallet_total_position", "filled_token_total_position", "filled_token_by_wallet_position", "wallet_trade_pnl",
]
available = [c for c in display_cols if c in event_ledger.columns]

# Show a few trigger+fill pairs
sample_orders = event_ledger["order_id"].unique()[:5]
event_ledger[
    event_ledger["order_id"].isin(sample_orders)
][available].sort_values(["order_id", "row_type"])


,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,...,qty_ordered,qty_filled,fill_qty,fill_source,token_winner,fill_pnl_usdc,filled_wallet_total_position,filled_token_total_position,filled_token_by_wallet_position,wallet_trade_pnl
0,trigger,1,0xaaa2cde772ab98ea86c9c323802f96daf03ec75f,0xfbeac45ae53828674baf51e12313b7fb651594915cfb082e7cc8694896f6fc21,Yes,BUY,2026-04-16 00:40:20+00:00,0xd535a7b09a16601283a1eefaa8bcc17c5769de6cf19faefb92ec9242799be003,0.710,0.710,...,14.084507,0.000000,NaN,None,True,NaN,0.000000,0.000000,0.000000,11.60000
1,trigger,2,0xaaa2cde772ab98ea86c9c323802f96daf03ec75f,0xef29f025924eca522d0662c20c7787e48922ffe72530348a49da47915c35d597,Yes,BUY,2026-04-16 01:25:04+00:00,0xce245795b069a82a7f7684fc674c0706ff740ac5267e0b8bee3e987a9e6e0f64,0.640,0.640,...,15.625000,0.000000,NaN,None,False,NaN,0.000000,0.000000,0.000000,-32.00000
3,fill,3,0x22dbd51e1a4e10fff072fa24300238c64033190f,0x6d64853f670b85151d2441b2bdea142f2031b573947ab2887e82585546fdc7f4,No,BUY,2026-04-16 01:52:00+00:00,0x55e617b3156c9a4f09baeab2d6c7e7b7725ec31acc6bb0418a05f930f0954072,0.850,NaN,...,11.764706,NaN,3.517646,same_token,True,0.524657,3.517646,3.517646,3.517646,NaN
4,fill,3,0x22dbd51e1a4e10fff072fa24300238c64033190f,0x6d64853f670b85151d2441b2bdea142f2031b573947ab2887e82585546fdc7f4,No,BUY,2026-04-16 01:52:02+00:00,0x8fe532b22ad24b9eae60c30a343ffb4335f7db5b22f0d740740f935f4d319e43,0.850,NaN,...,11.764706,NaN,2.352940,same_token,True,0.350941,5.870586,5.870586,5.870586,NaN
6,fill,3,0x22dbd51e1a4e10fff072fa24300238c64033190f,0x6d64853f670b85151d2441b2bdea142f2031b573947ab2887e82585546fdc7f4,No,BUY,2026-04-16 01:52:18+00:00,0x9246161cb4d5f869dc917aff6194e70e198a4fa853672a1420d7c69ec419aefd,0.840,NaN,...,11.764706,NaN,5.894120,same_token,True,0.879108,11.764706,11.764706,11.764706,NaN
2,trigger,3,0x22dbd51e1a4e10fff072fa24300238c64033190f,0x6d64853f670b85151d2441b2bdea142f2031b573947ab2887e82585546fdc7f4,No,BUY,2026-04-16 01:51:52+00:00,0xc6504e9e254cc462d5a26a117a5811b29e19c6d4f7d021b3e4ce45e85eb7630f,0.850,0.850,...,11.764706,11.764706,NaN,None,True,NaN,0.000000,0.000000,0.000000,30.00000
7,fill,4,0x22dbd51e1a4e10fff072fa24300238c64033190f,0x531afb8dafa0c026464827a4d6c4260743f6d2908101ae43645d6513e6bc5438,No,BUY,2026-04-16 01:56:04+00:00,0xe71302c7ccc2c8eae820f48e6e067b557e811a7ceb68a7a1561be6d370dafad8,0.997,NaN,...,10.030090,NaN,6.170000,same_token,True,0.012359,6.170000,6.170000,6.170000,NaN
5,trigger,4,0x22dbd51e1a4e10fff072fa24300238c64033190f,0x531afb8dafa0c026464827a4d6c4260743f6d2908101ae43645d6513e6bc5438,No,BUY,2026-04-16 01:52:14+00:00,0x5d5233efc3aba3bfc75dfa43b82313e23aad08f389fdf754cf9c4ac9507220f4,0.997,0.997,...,10.030090,6.170000,NaN,None,True,NaN,0.000000,0.000000,0.000000,0.03030
8,trigger,5,0x22dbd51e1a4e10fff072fa24300238c64033190f,0x531afb8dafa0c026464827a4d6c4260743f6d2908101ae43645d6513e6bc5438,No,BUY,2026-04-16 01:56:04+00:00,0xe71302c7ccc2c8eae820f48e6e067b557e811a7ceb68a7a1561be6d370dafad8,0.997,0.997,...,6.170000,0.000000,NaN,None,True,NaN,6.170000,6.170000,6.170000,0.01851


In [53]:
event_ledger[event_ledger['side'] == 'BUY'].groupby('wallet').agg(
    fill_pnl_usdc_sum=('fill_pnl_usdc', 'sum'),
    wallet_trade_pnl_sum=('wallet_trade_pnl', 'sum'),
    buy_count=('side', 'count')
)

,fill_pnl_usdc_sum,wallet_trade_pnl_sum,buy_count
wallet,,,
0x0cb10c40b0776e9ee8cef970af85724654dda76c,-42.046447,1672.790363,1048
0x0dedae6a02ea2ff8018ba5f277632919ed1c9025,125.633199,1754.770566,193
0x22dbd51e1a4e10fff072fa24300238c64033190f,-141.468184,10518.217308,2581
0x30f7b31ab5f1a3d98de5492df4d0f9110f63908e,0.000028,26.353410,38
0x3e438fdb437d90925b47b6f8ab85405ba8ffa83f,1.366515,112.418815,15
0x5042e6dc8a612c493881a3e67519cc09f5f4fcb0,0.835968,611.931374,401
0x672225e5e1aba1c970ac613efd1505f4b7a10762,-192.131672,-1301.137931,540
0x6bc74c392c320cfe10d5be61db978a58c8444ad4,222.537871,4454.873056,256
0x9783178832982d420baa733d2ec29b020eb9264f,-38.782067,-67.330830,743


In [54]:
event_ledger[event_ledger['side'] == 'BUY'].groupby('condition_id').agg(
    fill_pnl_usdc_sum=('fill_pnl_usdc', 'sum'),
    wallet_trade_pnl_sum=('wallet_trade_pnl', 'sum'),
    buy_count=('side', 'count')
).sort_values('fill_pnl_usdc_sum', ascending=False).head(10)

,fill_pnl_usdc_sum,wallet_trade_pnl_sum,buy_count
condition_id,,,
0x5341e90b72330a8eb96cb685dcffe51ad41783cc12e91b6858b9bd996b3d5b63,134.621843,1372.997496,67
0xdda63e9a4419b36eed1559287ec645677d23cb95d96ce7d5c91d83cb6a8b268e,89.990000,359.991999,9
0xa743ce4a2a42aa232c2caa41aa27837ff1cde07781a1e9a726342065c3d86ba6,76.076351,1300.499003,43
0xaa145cf5714455f546afe53037b8712df749ba96c04820c2d41f37f877f29697,67.838875,1451.789583,48
0xb7dd41c16cd5b59e543ffcdab6a0d876f5ccef5329ea42ef1798d80c9a2b8499,59.880973,-666.691986,42
0x2c32d0c0056fd2650a39549f6382ab0ed7953fddaf8fcf6875b6b1451d6680b0,52.396067,160.699000,12
0xc6c25c2150b2e2c7758ee0ad8640324f60086367fc689f0bca618621c890a2c6,52.042117,320.522167,19
0x4843f853a4e9e718c0660449db0c956115aca8e2c20c8e41833a95d7a4d32193,48.813529,120.702800,8
0xbdf0bbe0d1fd006aec6deef5482ca94b8ff0ceea73860bda0a337c4ce50d4448,45.616651,89.000000,5


In [55]:
(fills[fills['condition_id'] == '0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86e6953305a2ef19c77f20'][['dt', 'side', 'outcome',
                                                                                                        'price', 'fill_qty', 'exec_price', 
                                                                                                        # 'filled_wallet_total_position',
                                                                                                          'filled_token_total_position', 
                                                                                                          'filled_token_by_wallet_position', 
                                                                                                          'fill_pnl_usdc',
                                                                                                          'order_id','wallet', 'tx_hash']]
 .sort_values('dt')
 .head(10))

,dt,side,outcome,price,fill_qty,exec_price,filled_token_total_position,filled_token_by_wallet_position,fill_pnl_usdc,order_id,wallet,tx_hash


In [56]:
sample_filled_orders = fills["order_id"].unique()[:50]
event_ledger[
    (event_ledger['side'] == 'SELL') &
    (event_ledger["order_id"].isin(sample_filled_orders))
].sort_values(["order_id", "dt"])

,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,...,fill_qty,fill_source,token_winner,exec_price,fill_pnl_usdc,wallet_trade_pnl,filled_wallet_total_position,filled_token_total_position,filled_token_by_wallet_position,shard
83,trigger,42,0x22dbd51e1a4e10fff072fa24300238c64033190f,0xc7140ddb5ae5dc94d4553fb05d4600816f33ff024844cebe8326f4c41c4a1a47,No,SELL,2026-04-16 12:04:32+00:00,0x80d389efc8a3261ed0f78deac4853f59386c417d31e0e8cc164c1e712c6d1eb6,0.436000,0.436000,...,NaN,None,False,NaN,NaN,174.400000,100.000000,88.885392,88.885392,c
84,fill,42,0x22dbd51e1a4e10fff072fa24300238c64033190f,0xc7140ddb5ae5dc94d4553fb05d4600816f33ff024844cebe8326f4c41c4a1a47,No,SELL,2026-04-16 12:04:36+00:00,0x5613da9fc95ff34e9d9c0ce3da13a1729bc5a212b4ac85e3e8409e505f8d8cfd,0.436000,NaN,...,17.730496,same_token,False,0.436000,7.722766,NaN,82.269504,71.154896,71.154896,c
114,trigger,67,0x22dbd51e1a4e10fff072fa24300238c64033190f,0xc7140ddb5ae5dc94d4553fb05d4600816f33ff024844cebe8326f4c41c4a1a47,No,SELL,2026-04-16 14:41:08+00:00,0x455bdad1195aba9d14e0512ee19745f33f67a8d7b4f7b51c2f66484149bc51ce,0.201364,0.201364,...,NaN,None,False,NaN,NaN,40.272820,100.000000,88.885392,88.885392,c
115,fill,67,0x22dbd51e1a4e10fff072fa24300238c64033190f,0xc7140ddb5ae5dc94d4553fb05d4600816f33ff024844cebe8326f4c41c4a1a47,No,SELL,2026-04-16 14:41:14+00:00,0xceb7ba9fec886444df72112bc4a1bb78534a9e310aa6f6bbb76514d4618b6ad0,0.209000,NaN,...,10.000000,opposite_token,False,0.201364,2.011627,NaN,90.000000,78.885392,78.885392,c
116,fill,67,0x22dbd51e1a4e10fff072fa24300238c64033190f,0xc7140ddb5ae5dc94d4553fb05d4600816f33ff024844cebe8326f4c41c4a1a47,No,SELL,2026-04-16 14:41:28+00:00,0xc556d19f1eb41dd508df427f674a3901dac2eba81f1be275b6806f514d5b66f9,0.210000,NaN,...,2.521350,same_token,False,0.201364,0.507202,NaN,87.478650,76.364042,76.364042,c
117,trigger,68,0x22dbd51e1a4e10fff072fa24300238c64033190f,0xc7140ddb5ae5dc94d4553fb05d4600816f33ff024844cebe8326f4c41c4a1a47,No,SELL,2026-04-16 14:41:38+00:00,0xc0eeeaf88160b84de3a010023517403be7035d524b7dd3da018d92ddee4ba5f1,0.209000,0.209000,...,NaN,None,False,NaN,NaN,1.051270,87.478650,76.364042,76.364042,c
118,fill,68,0x22dbd51e1a4e10fff072fa24300238c64033190f,0xc7140ddb5ae5dc94d4553fb05d4600816f33ff024844cebe8326f4c41c4a1a47,No,SELL,2026-04-16 14:41:42+00:00,0x16d1327c21658596bf329739ac92f49d1d0f6c0dc7926b54a451d5e1dd9e6aad,0.209000,NaN,...,5.030000,same_token,False,0.209000,1.050219,NaN,82.448650,71.334042,71.334042,c
119,trigger,69,0x22dbd51e1a4e10fff072fa24300238c64033190f,0xc7140ddb5ae5dc94d4553fb05d4600816f33ff024844cebe8326f4c41c4a1a47,No,SELL,2026-04-16 14:41:42+00:00,0x16d1327c21658596bf329739ac92f49d1d0f6c0dc7926b54a451d5e1dd9e6aad,0.209000,0.209000,...,NaN,None,False,NaN,NaN,61.648730,82.448650,71.334042,71.334042,c
120,fill,69,0x22dbd51e1a4e10fff072fa24300238c64033190f,0xc7140ddb5ae5dc94d4553fb05d4600816f33ff024844cebe8326f4c41c4a1a47,No,SELL,2026-04-16 14:42:02+00:00,0x1140abc00b12107eee001efb0480c3d9972b5dd030e4530a701e6d67d9f77101,0.231000,NaN,...,10.210000,same_token,False,0.209000,2.131756,NaN,72.238650,61.124042,61.124042,c
121,fill,69,0x22dbd51e1a4e10fff072fa24300238c64033190f,0xc7140ddb5ae5dc94d4553fb05d4600816f33ff024844cebe8326f4c41c4a1a47,No,SELL,2026-04-16 14:42:02+00:00,0x1140abc00b12107eee001efb0480c3d9972b5dd030e4530a701e6d67d9f77101,0.231000,NaN,...,2.432225,same_token,False,0.209000,0.507827,NaN,69.806425,58.691816,58.691816,c


## Build PnL time series

In [57]:
def resample_hourly_series(df: pd.DataFrame, dt_col: str, pnl_col: str) -> pd.DataFrame:
    """Resample a PnL column to 1-h buckets and compute cumulative sum."""
    if df.empty or pnl_col not in df.columns:
        return pd.DataFrame(columns=["trade_dt", "net_pnl_usdc", "cum_net_pnl_usdc"])
    hourly = (
        df.assign(trade_dt=df[dt_col].dt.floor("1h"))
        .groupby("trade_dt", as_index=False)[pnl_col]
        .sum()
        .rename(columns={pnl_col: "net_pnl_usdc"})
        .sort_values("trade_dt")
        .reset_index(drop=True)
    )
    hourly["cum_net_pnl_usdc"] = hourly["net_pnl_usdc"].cumsum()
    return hourly


def with_zero_anchor(hourly: pd.DataFrame) -> pd.DataFrame:
    """Prepend a zero anchor one hour before the first bucket."""
    if hourly.empty:
        return hourly
    anchor = pd.DataFrame({
        "trade_dt": [hourly["trade_dt"].min() - pd.Timedelta(hours=1)],
        "net_pnl_usdc": [0.0],
        "cum_net_pnl_usdc": [0.0],
    })
    return pd.concat([anchor, hourly], ignore_index=True)


# Wallet cohort PnL: from trigger rows (their actual trade_pnl)
wallet_hourly = resample_hourly_series(triggers, "dt", "wallet_trade_pnl")

# Copy-strategy PnL: from fill rows
copy_hourly = resample_hourly_series(fills, "dt", "fill_pnl_usdc")

print(f"Wallet cohort hourly buckets : {len(wallet_hourly)}")
print(f"Copy strategy hourly buckets : {len(copy_hourly)}")

Wallet cohort hourly buckets : 829
Copy strategy hourly buckets : 381


In [58]:
wallet_hourly.head()

,trade_dt,net_pnl_usdc,cum_net_pnl_usdc
0,2026-04-16 00:00:00+00:00,11.60000,11.60000
1,2026-04-16 01:00:00+00:00,-1.95119,9.64881
2,2026-04-16 02:00:00+00:00,67.18580,76.83461
3,2026-04-16 03:00:00+00:00,4.69940,81.53401
4,2026-04-16 04:00:00+00:00,0.02000,81.55401


## Cumulative PnL chart

In [59]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = go.Figure()

# ── Wallet cohort trace ───────────────────────────────────────────────────────
if not wallet_hourly.empty:
    wh = with_zero_anchor(wallet_hourly)
    fig.add_trace(go.Scatter(
        x=wh["trade_dt"],
        y=wh["cum_net_pnl_usdc"],
        mode="lines",
        line=dict(dash="dot", width=2),
        opacity=0.7,
        name="Watched wallets (raw PnL)",
        hovertemplate=(
            "<b>Watched wallets</b><br>"
            "%{x|%Y-%m-%d %H:%M}<br>"
            "cum PnL: %{y:.2f} USDC<extra></extra>"
        ),
    ))

# ── Copy-strategy trace ───────────────────────────────────────────────────────
if not copy_hourly.empty:
    ch = with_zero_anchor(copy_hourly)
    fig.add_trace(go.Scatter(
        x=ch["trade_dt"],
        y=ch["cum_net_pnl_usdc"],
        mode="lines",
        line=dict(dash="solid", width=2),
        name="Copy strategy (filled)",
        hovertemplate=(
            "<b>Copy strategy</b><br>"
            "%{x|%Y-%m-%d %H:%M}<br>"
            "cum PnL: %{y:.2f} USDC<extra></extra>"
        ),
    ))

fig.add_hline(y=0, line_dash="dash", line_color="grey", line_width=1, opacity=0.5)

fig.update_layout(
    template="plotly_white",
    height=480,
    title=dict(
        text=(
            f"Copy-trade backtest — cumulative PnL (1h) | "
            f"{period_start} → {period_end} | "
            f"{len(WATCHED_WALLETS)} wallets | "
            f"worse_price={WORSE_PRICE_OFFSET:.2f} | "
            f"horizon={int(FILL_HORIZON_SECONDS)}s"
        ),
        font=dict(size=13),
    ),
    xaxis_title="Time (1h buckets)",
    yaxis_title="Cumulative net PnL (USDC)",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)

fig.show()

## Diagnostics

In [60]:
# Fill rate by side (order-level)
if not triggers.empty:
    trig_by_side = triggers.groupby("side").size().rename("triggers")
    fill_by_side = fills.groupby("side")["order_id"].nunique().rename("orders_filled")
    fill_rows_by_side = fills.groupby("side").size().rename("fill_rows")
    side_summary = pd.concat([trig_by_side, fill_by_side, fill_rows_by_side], axis=1).fillna(0).astype(int)
    side_summary["fill_rate"] = (side_summary["orders_filled"] / side_summary["triggers"] * 100).round(1)
    display(side_summary)


,triggers,orders_filled,fill_rows,fill_rate
side,,,,
BUY,5012,989,1593,19.7
SELL,534,384,575,71.9


In [61]:
# Fill source breakdown by side
if not fills.empty:
    display(
        fills.groupby(["side", "fill_source"]).size()
        .rename("count")
        .reset_index()
        .sort_values(["side", "fill_source"])
    )

,side,fill_source,count
0,BUY,opposite_token,403
1,BUY,same_token,1190
2,SELL,opposite_token,211
3,SELL,same_token,364


In [62]:
df = event_ledger.assign(
    is_trigger=event_ledger["row_type"].eq("trigger"),
    is_fill=event_ledger["row_type"].eq("fill"),
)

# --- wallet-level stats (row-based) ---
wallet_stats = (
    df.groupby("wallet")
    .agg(
        total_triggers=("is_trigger", "sum"),
        total_fills=("is_fill", "sum"),
        total_fill_pnl=("fill_pnl_usdc", "sum"),
        total_trade_pnl=("wallet_trade_pnl", "sum"),
        total_pnl=("fill_pnl_usdc", "sum"),
    )
)

# --- order-level logic (trigger -> fill) ---
order_stats = (
    df.groupby(["wallet", "order_id"])
    .agg(
        has_trigger=("is_trigger", "any"),
        has_fill=("is_fill", "any"),
    )
    .assign(trigger_with_fill=lambda x: x["has_trigger"] & x["has_fill"])
    .groupby("wallet")
    .agg(triggers_with_fill=("trigger_with_fill", "sum"))
)

# --- combine ---
result = wallet_stats.join(order_stats)

# --- derived metric ---
result["fill_ratio"] = (
    result["triggers_with_fill"] / result["total_triggers"]
).fillna(0)

result.sort_values("total_pnl", ascending=False).head()

,total_triggers,total_fills,total_fill_pnl,total_trade_pnl,total_pnl,triggers_with_fill,fill_ratio
wallet,,,,,,,
0x6bc74c392c320cfe10d5be61db978a58c8444ad4,172,154,272.914604,9523.415566,272.914604,83,0.482558
0x9b2fcb79bdb8a6a6174da327e9335f7889c6d8f8,403,286,120.198825,972.942983,120.198825,170,0.421836
0x0dedae6a02ea2ff8018ba5f277632919ed1c9025,168,39,106.441293,1667.136219,106.441293,26,0.154762
0xaaa2cde772ab98ea86c9c323802f96daf03ec75f,196,56,36.974614,182.362698,36.974614,38,0.193878
0x22dbd51e1a4e10fff072fa24300238c64033190f,2270,748,22.724793,11252.849959,22.724793,469,0.206608


In [63]:
len(df)

7714

In [64]:
# get diff between trigger and fill timestamp per order_id
# without lambda, using groupby + agg + merge
trigger_df = df[df['row_type'] == 'trigger'].groupby('order_id')['dt'].min().reset_index()
fill_df = df[df['row_type'] == 'fill'].groupby('order_id')['dt'].min().reset_index()

merged_df = pd.merge(trigger_df, fill_df, on='order_id', how='outer', suffixes=('_trigger', '_fill'))
merged_df['diff_dt'] = merged_df['dt_fill'] - merged_df['dt_trigger']

In [65]:
result[result['total_fill_pnl'] < 0].sort_values("total_fill_pnl").index.tolist()

['0x9783178832982d420baa733d2ec29b020eb9264f']

In [66]:
# Per-wallet PnL breakdown
if not fills.empty:
    wallet_pnl_df = (
        fills.groupby("wallet")
        .agg(
            filled_orders=("order_id", "nunique"),
            fill_rows=("order_id", "count"),
            net_pnl_usdc=("fill_pnl_usdc", "sum"),
            total_qty=("fill_qty", "sum"),
            win_rate=("token_winner", lambda s: fills.loc[s.index].groupby("order_id")["token_winner"].first().mean()),
        )
        .assign(win_rate=lambda d: (d["win_rate"] * 100).round(1))
        .sort_values("net_pnl_usdc", ascending=False)
        .reset_index()
    )
    display(wallet_pnl_df)


,wallet,filled_orders,fill_rows,net_pnl_usdc,total_qty,win_rate
0,0x6bc74c392c320cfe10d5be61db978a58c8444ad4,83,154,272.914604,1775.482764,50.6
1,0x9b2fcb79bdb8a6a6174da327e9335f7889c6d8f8,170,286,120.198825,2846.482634,62.9
2,0x0dedae6a02ea2ff8018ba5f277632919ed1c9025,26,39,106.441293,473.221132,76.9
3,0xaaa2cde772ab98ea86c9c323802f96daf03ec75f,38,56,36.974614,419.122710,71.1
4,0x22dbd51e1a4e10fff072fa24300238c64033190f,469,748,22.724793,5790.429267,55.2
5,0x0cb10c40b0776e9ee8cef970af85724654dda76c,233,357,13.783785,3475.282024,33.9
6,0x672225e5e1aba1c970ac613efd1505f4b7a10762,152,228,6.950085,2060.942941,49.3
7,0xb78ddd792ad7d1f603fcd046c662d6d9adbf835d,6,12,3.959688,152.147980,66.7
8,0x3e438fdb437d90925b47b6f8ab85405ba8ffa83f,3,3,1.366515,31.396515,100.0
9,0x5042e6dc8a612c493881a3e67519cc09f5f4fcb0,40,48,0.608327,427.381641,80.0


In [67]:
# Orders that were NOT filled at all
filled_order_ids = set(fills["order_id"].unique()) if not fills.empty else set()
unfilled_triggers = triggers[~triggers["order_id"].isin(filled_order_ids)]
print(f"Unfilled orders    : {len(unfilled_triggers):,} ({len(unfilled_triggers)/max(len(triggers),1)*100:.1f}% of all triggers)")

# Partially filled orders (filled but qty_filled < qty_ordered)
if not fills.empty:
    filled_trig = triggers[triggers["order_id"].isin(filled_order_ids)].copy()
    partial_mask = filled_trig["qty_filled"] < filled_trig["qty_ordered"] - 1e-9
    partial_fills = filled_trig[partial_mask]
    print(f"Partially filled   : {len(partial_fills):,} ({len(partial_fills)/max(len(filled_trig),1)*100:.1f}% of filled orders)")
    print(f"Fully filled       : {len(filled_trig)-len(partial_fills):,}")

if not unfilled_triggers.empty:
    print("\nUnfilled breakdown by side:")
    display(unfilled_triggers.groupby("side").size().rename("count").reset_index())


Unfilled orders    : 4,173 (75.2% of all triggers)
Partially filled   : 304 (22.1% of filled orders)
Fully filled       : 1,069

Unfilled breakdown by side:


,side,count
0,BUY,4023
1,SELL,150


In [68]:
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)
fills[fills['order_id'] == 34]

,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,...,fill_qty,fill_source,token_winner,exec_price,fill_pnl_usdc,wallet_trade_pnl,filled_wallet_total_position,filled_token_total_position,filled_token_by_wallet_position,shard
56,fill,34,0x22dbd51e1a4e10fff072fa24300238c64033190f,0xc7140ddb5ae5dc94d4553fb05d4600816f33ff024844cebe8326f4c41c4a1a47,No,BUY,2026-04-16 10:48:56+00:00,0x69b0d19dadf0d64b2e07fa5c0c8ee12f299a1b1799b4d006c3f0bcabf2382ca7,0.341,NaN,...,14.270000,same_token,False,0.341,-4.870936,NaN,41.694995,41.694995,41.694995,c
59,fill,34,0x22dbd51e1a4e10fff072fa24300238c64033190f,0xc7140ddb5ae5dc94d4553fb05d4600816f33ff024844cebe8326f4c41c4a1a47,No,BUY,2026-04-16 10:48:58+00:00,0xa197bc0b45ad60cf420bd1009cf56254b5f14c6d59e9e87cb8282d8bcec417ee,0.341,NaN,...,4.640000,same_token,False,0.341,-1.583822,NaN,46.334995,46.334995,46.334995,c
60,fill,34,0x22dbd51e1a4e10fff072fa24300238c64033190f,0xc7140ddb5ae5dc94d4553fb05d4600816f33ff024844cebe8326f4c41c4a1a47,No,BUY,2026-04-16 10:48:58+00:00,0xa197bc0b45ad60cf420bd1009cf56254b5f14c6d59e9e87cb8282d8bcec417ee,0.340,NaN,...,9.003392,opposite_token,False,0.341,-3.073227,NaN,55.338387,55.338387,55.338387,c
61,fill,34,0x22dbd51e1a4e10fff072fa24300238c64033190f,0xc7140ddb5ae5dc94d4553fb05d4600816f33ff024844cebe8326f4c41c4a1a47,No,BUY,2026-04-16 10:49:02+00:00,0xa241658dd116f0d0d1eb94fb5306ef42747af3004ff58d0cc62fc500c92d9bd5,0.340,NaN,...,1.412121,opposite_token,False,0.341,-0.482015,NaN,56.750508,56.750508,56.750508,c


In [69]:
fills.groupby("wallet")["fill_pnl_usdc"].sum().sort_values(ascending=False, key=abs).head(100)

wallet
0x6bc74c392c320cfe10d5be61db978a58c8444ad4    272.914604
0x9b2fcb79bdb8a6a6174da327e9335f7889c6d8f8    120.198825
0x0dedae6a02ea2ff8018ba5f277632919ed1c9025    106.441293
0x9783178832982d420baa733d2ec29b020eb9264f    -39.854570
0xaaa2cde772ab98ea86c9c323802f96daf03ec75f     36.974614
0x22dbd51e1a4e10fff072fa24300238c64033190f     22.724793
0x0cb10c40b0776e9ee8cef970af85724654dda76c     13.783785
0x672225e5e1aba1c970ac613efd1505f4b7a10762      6.950085
0xb78ddd792ad7d1f603fcd046c662d6d9adbf835d      3.959688
0x3e438fdb437d90925b47b6f8ab85405ba8ffa83f      1.366515
0x5042e6dc8a612c493881a3e67519cc09f5f4fcb0      0.608327
0x30f7b31ab5f1a3d98de5492df4d0f9110f63908e      0.000028
Name: fill_pnl_usdc, dtype: float64

In [70]:
trigger_wallets = triggers[["order_id", "wallet"]].set_index("order_id")["wallet"]

fills_with_wallet = fills.merge(trigger_wallets, on="order_id", how="left", suffixes=("", "_trigger"))

fills_with_wallet['notional'] = fills_with_wallet['fill_qty'] * fills_with_wallet['exec_price']

fills_with_wallet.groupby(["wallet", "condition_id"]).agg(
    pnl=("fill_pnl_usdc", "sum"),
    orders_filled=("order_id", "nunique")
    ).sort_values(['wallet', "pnl"], ascending=[True, False]).head(1)

,,pnl,orders_filled
wallet,condition_id,,
0x0cb10c40b0776e9ee8cef970af85724654dda76c,0x4843f853a4e9e718c0660449db0c956115aca8e2c20c8e41833a95d7a4d32193,48.813529,1


In [71]:
fills_with_wallet.head()

,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,...,token_winner,exec_price,fill_pnl_usdc,wallet_trade_pnl,filled_wallet_total_position,filled_token_total_position,filled_token_by_wallet_position,shard,wallet_trigger,notional
0,fill,3,0x22dbd51e1a4e10fff072fa24300238c64033190f,0x6d64853f670b85151d2441b2bdea142f2031b573947ab2887e82585546fdc7f4,No,BUY,2026-04-16 01:52:00+00:00,0x55e617b3156c9a4f09baeab2d6c7e7b7725ec31acc6bb0418a05f930f0954072,0.850,NaN,...,True,0.850,0.524657,NaN,3.517646,3.517646,3.517646,6,0x22dbd51e1a4e10fff072fa24300238c64033190f,2.989999
1,fill,3,0x22dbd51e1a4e10fff072fa24300238c64033190f,0x6d64853f670b85151d2441b2bdea142f2031b573947ab2887e82585546fdc7f4,No,BUY,2026-04-16 01:52:02+00:00,0x8fe532b22ad24b9eae60c30a343ffb4335f7db5b22f0d740740f935f4d319e43,0.850,NaN,...,True,0.850,0.350941,NaN,5.870586,5.870586,5.870586,6,0x22dbd51e1a4e10fff072fa24300238c64033190f,1.999999
2,fill,3,0x22dbd51e1a4e10fff072fa24300238c64033190f,0x6d64853f670b85151d2441b2bdea142f2031b573947ab2887e82585546fdc7f4,No,BUY,2026-04-16 01:52:18+00:00,0x9246161cb4d5f869dc917aff6194e70e198a4fa853672a1420d7c69ec419aefd,0.840,NaN,...,True,0.850,0.879108,NaN,11.764706,11.764706,11.764706,6,0x22dbd51e1a4e10fff072fa24300238c64033190f,5.010002
3,fill,4,0x22dbd51e1a4e10fff072fa24300238c64033190f,0x531afb8dafa0c026464827a4d6c4260743f6d2908101ae43645d6513e6bc5438,No,BUY,2026-04-16 01:56:04+00:00,0xe71302c7ccc2c8eae820f48e6e067b557e811a7ceb68a7a1561be6d370dafad8,0.997,NaN,...,True,0.997,0.012359,NaN,6.170000,6.170000,6.170000,5,0x22dbd51e1a4e10fff072fa24300238c64033190f,6.151490
4,fill,6,0x22dbd51e1a4e10fff072fa24300238c64033190f,0x531afb8dafa0c026464827a4d6c4260743f6d2908101ae43645d6513e6bc5438,No,BUY,2026-04-16 02:06:22+00:00,0x03bf0aa1c3688205bc05f3129601b900016d2da940c0363cd3e3a2195cc1b2bd,0.997,NaN,...,True,0.997,0.020090,NaN,16.200090,16.200090,16.200090,5,0x22dbd51e1a4e10fff072fa24300238c64033190f,10.000000


In [72]:
wallet_pnls = (
    fills_with_wallet
    .groupby(["wallet", "condition_id"])
    .agg(
        pnl=("fill_pnl_usdc", "sum"),
        orders_filled=("order_id", "nunique"),
        notional=("notional", "sum"),
    )
    .groupby("wallet")
    .agg(
        pnl=("pnl", "sum"),
        notional=("notional", "sum"),
        unique_conditions=("pnl", "size"),  # ✅ count rows
        total_orders_filled=("orders_filled", "sum"),
    )
    .sort_values("pnl", ascending=False)
)

In [73]:
wallet_pnls.head()

,pnl,notional,unique_conditions,total_orders_filled
wallet,,,,
0x6bc74c392c320cfe10d5be61db978a58c8444ad4,272.914604,736.571798,18,83
0x9b2fcb79bdb8a6a6174da327e9335f7889c6d8f8,120.198825,1360.217378,53,170
0x0dedae6a02ea2ff8018ba5f277632919ed1c9025,106.441293,144.206406,17,26
0xaaa2cde772ab98ea86c9c323802f96daf03ec75f,36.974614,170.090732,17,38
0x22dbd51e1a4e10fff072fa24300238c64033190f,22.724793,3322.704517,79,469


In [74]:
MAX_WALLET_EXPOSURE = 1000.0  # USDC
wallet_pnls['pnl_limited'] = np.where(
    wallet_pnls['notional'] <= MAX_WALLET_EXPOSURE,
    wallet_pnls['pnl'],
    wallet_pnls['pnl'] * MAX_WALLET_EXPOSURE / wallet_pnls['notional']
)

In [75]:
wallet_pnls.head(10)

,pnl,notional,unique_conditions,total_orders_filled,pnl_limited
wallet,,,,,
0x6bc74c392c320cfe10d5be61db978a58c8444ad4,272.914604,736.571798,18,83,272.914604
0x9b2fcb79bdb8a6a6174da327e9335f7889c6d8f8,120.198825,1360.217378,53,170,88.367365
0x0dedae6a02ea2ff8018ba5f277632919ed1c9025,106.441293,144.206406,17,26,106.441293
0xaaa2cde772ab98ea86c9c323802f96daf03ec75f,36.974614,170.090732,17,38,36.974614
0x22dbd51e1a4e10fff072fa24300238c64033190f,22.724793,3322.704517,79,469,6.839246
0x0cb10c40b0776e9ee8cef970af85724654dda76c,13.783785,992.521908,43,233,13.783785
0x672225e5e1aba1c970ac613efd1505f4b7a10762,6.950085,1116.745386,31,152,6.223518
0xb78ddd792ad7d1f603fcd046c662d6d9adbf835d,3.959688,38.291987,4,6,3.959688
0x3e438fdb437d90925b47b6f8ab85405ba8ffa83f,1.366515,30.000000,2,3,1.366515


In [76]:
result = (
    fills_with_wallet
    # 1️⃣ PnL per market (condition) per wallet
    .groupby(["wallet", "condition_id"])["fill_pnl_usdc"]
    .sum()
    
    # 2️⃣ Then per wallet
    .groupby("wallet")
    .agg(
        unique_conditions="count",   # number of markets traded
        median_market_pnl="median",  # median PnL across markets
    )
    .sort_values("unique_conditions", ascending=False)
    .head(20)
)

In [77]:
result.sort_values("median_market_pnl", ascending=False).head(100)

,unique_conditions,median_market_pnl
wallet,,
0x0dedae6a02ea2ff8018ba5f277632919ed1c9025,17,3.496894
0x9b2fcb79bdb8a6a6174da327e9335f7889c6d8f8,53,1.645022
0xaaa2cde772ab98ea86c9c323802f96daf03ec75f,17,1.299300
0xb78ddd792ad7d1f603fcd046c662d6d9adbf835d,4,0.712976
0x3e438fdb437d90925b47b6f8ab85405ba8ffa83f,2,0.683258
0x5042e6dc8a612c493881a3e67519cc09f5f4fcb0,27,0.516316
0x22dbd51e1a4e10fff072fa24300238c64033190f,79,0.317296
0x9783178832982d420baa733d2ec29b020eb9264f,97,0.194082
0x672225e5e1aba1c970ac613efd1505f4b7a10762,31,0.002521


In [78]:
fills_with_wallet.head()

,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,...,token_winner,exec_price,fill_pnl_usdc,wallet_trade_pnl,filled_wallet_total_position,filled_token_total_position,filled_token_by_wallet_position,shard,wallet_trigger,notional
0,fill,3,0x22dbd51e1a4e10fff072fa24300238c64033190f,0x6d64853f670b85151d2441b2bdea142f2031b573947ab2887e82585546fdc7f4,No,BUY,2026-04-16 01:52:00+00:00,0x55e617b3156c9a4f09baeab2d6c7e7b7725ec31acc6bb0418a05f930f0954072,0.850,NaN,...,True,0.850,0.524657,NaN,3.517646,3.517646,3.517646,6,0x22dbd51e1a4e10fff072fa24300238c64033190f,2.989999
1,fill,3,0x22dbd51e1a4e10fff072fa24300238c64033190f,0x6d64853f670b85151d2441b2bdea142f2031b573947ab2887e82585546fdc7f4,No,BUY,2026-04-16 01:52:02+00:00,0x8fe532b22ad24b9eae60c30a343ffb4335f7db5b22f0d740740f935f4d319e43,0.850,NaN,...,True,0.850,0.350941,NaN,5.870586,5.870586,5.870586,6,0x22dbd51e1a4e10fff072fa24300238c64033190f,1.999999
2,fill,3,0x22dbd51e1a4e10fff072fa24300238c64033190f,0x6d64853f670b85151d2441b2bdea142f2031b573947ab2887e82585546fdc7f4,No,BUY,2026-04-16 01:52:18+00:00,0x9246161cb4d5f869dc917aff6194e70e198a4fa853672a1420d7c69ec419aefd,0.840,NaN,...,True,0.850,0.879108,NaN,11.764706,11.764706,11.764706,6,0x22dbd51e1a4e10fff072fa24300238c64033190f,5.010002
3,fill,4,0x22dbd51e1a4e10fff072fa24300238c64033190f,0x531afb8dafa0c026464827a4d6c4260743f6d2908101ae43645d6513e6bc5438,No,BUY,2026-04-16 01:56:04+00:00,0xe71302c7ccc2c8eae820f48e6e067b557e811a7ceb68a7a1561be6d370dafad8,0.997,NaN,...,True,0.997,0.012359,NaN,6.170000,6.170000,6.170000,5,0x22dbd51e1a4e10fff072fa24300238c64033190f,6.151490
4,fill,6,0x22dbd51e1a4e10fff072fa24300238c64033190f,0x531afb8dafa0c026464827a4d6c4260743f6d2908101ae43645d6513e6bc5438,No,BUY,2026-04-16 02:06:22+00:00,0x03bf0aa1c3688205bc05f3129601b900016d2da940c0363cd3e3a2195cc1b2bd,0.997,NaN,...,True,0.997,0.020090,NaN,16.200090,16.200090,16.200090,5,0x22dbd51e1a4e10fff072fa24300238c64033190f,10.000000


In [79]:
from polymarket_analysis.data.data_catalogue import load_markets_processed
mdf = load_markets_processed()
mdf = mdf[
    ~(mdf['primary_tag'].isin(['Sports', 'Crypto']))
    & (mdf['winner_token_id'].notna())
    ].copy().reset_index(drop=True)


In [80]:
mdf.head()

,condition_id,end_date_iso,question,tags,primary_tag,winner_token_id,outcome
0,0x055176c9ebd8775c281ad540d5c16b3323e316507aef2d45c84f061cbc6bbcdc,2022-08-21T00:00:00Z,"MLB: Who will win Boston Red Sox v. Baltimore Orioles, scheduled for August 21, 7:10 PM ET?",[All],All,9612890763764062692282935414227141810568206972440321500296202304471805951204,Orioles
1,0x1409177eae7f1b0406bc0eea9d2715c9f76d88ec7765aed03cf39d59f36008f7,2023-01-06T00:00:00Z,Will a House Speaker be elected by end of Thursday?,[All],All,56419231299380534710656783098291368308638365867243518354348835265264811381897,No
2,0xd6165c1a30269d0799b27ee6ea90c048dfd6b27276cbd2aa7e18fe1c4562faa8,2023-01-14T00:00:00Z,Will Benjamin Netanyahu be the 2023 TIME Person of the Year?,"[Politics, Joe Biden, time magazine, person of the year, Awards, technology, news, future predic...",Politics,115040502429502340199300991115762950152511906206476760034830683863926486402417,No
3,0x9da76626846b7a0f7c5388feb280e42a67564ea104123020fbf59a641b179623,2023-01-14T00:00:00Z,Will Xi Jinping be the 2023 TIME Person of the Year?,"[Politics, Joe Biden, time magazine, person of the year, Awards, technology, news, future predic...",Politics,55070293174349524004776076543655661478039361630374630352242580072992541740735,No
4,0xe7d21325e5cfccbdaddf6ab6ef9bf477cf3aa6615dd36c8c408da99a1dd83237,2023-01-14T00:00:00Z,Will Volodymyr Zelenskyy be the 2023 TIME Person of the Year?,"[Politics, Joe Biden, time magazine, person of the year, Awards, technology, news, future predic...",Politics,5775128592397269721112080310923554806977502333079078864830955469822139503940,No


## Browser plots: PnL and positions

Interactive Plotly charts for cumulative PnL and signed positions. Market hovers include `end_date_iso` from `mdf`.


In [81]:
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = "browser"

TOP_WALLETS_PNL = 10
TOP_MARKETS_PNL = 10
TOP_WALLETS_POSITION = 10
TOP_MARKETS_POSITION = 10

def _short_wallet(wallet: str) -> str:
    wallet = str(wallet)
    return f"{wallet[:6]}...{wallet[-4:]}" if len(wallet) > 14 else wallet

def _short_text(text: str, width: int = 72) -> str:
    if pd.isna(text):
        return "Unknown market"
    text = " ".join(str(text).split())
    return text if len(text) <= width else text[: width - 3] + "..."

def _build_total_ts(df: pd.DataFrame, value_col: str, net_col: str, cum_col: str) -> pd.DataFrame:
    total = (
        df.sort_values("dt")
        .groupby("dt", as_index=False)[value_col]
        .sum()
        .rename(columns={value_col: net_col})
    )
    total[cum_col] = total[net_col].cumsum()
    return total

def _build_position_snapshots(df: pd.DataFrame) -> pd.DataFrame:
    series_cols = [
        "wallet",
        "wallet_label",
        "condition_id",
        "market_label",
        "end_date_iso",
        "market_close_dt",
        "outcome",
    ]

    active = df[
        df["market_close_dt"].isna() | (df["dt"] < df["market_close_dt"])
    ].copy()

    snapshots = (
        active.sort_values(series_cols + ["dt", "order_id"])
        .groupby(series_cols + ["dt"], as_index=False, dropna=False)
        .agg(
            token_position=("filled_token_by_wallet_position", "last"),
            exec_price=("exec_price", "last"),
        )
    )
    snapshots["usdc_position"] = snapshots["token_position"] * snapshots["exec_price"]

    if snapshots.empty:
        snapshots["token_delta"] = pd.Series(dtype=float)
        snapshots["usdc_delta"] = pd.Series(dtype=float)
        return snapshots

    reset = (
        snapshots.groupby(series_cols, as_index=False, dropna=False)
        .agg(
            token_position=("token_position", "last"),
            usdc_position=("usdc_position", "last"),
        )
    )
    reset = reset[
        reset["market_close_dt"].notna()
        & ((reset["token_position"].abs() > 1e-12) | (reset["usdc_position"].abs() > 1e-12))
    ].copy()
    reset["dt"] = reset["market_close_dt"]
    reset["token_position"] = 0.0
    reset["usdc_position"] = 0.0

    combined = pd.concat(
        [
            snapshots[series_cols + ["dt", "token_position", "usdc_position"]],
            reset[series_cols + ["dt", "token_position", "usdc_position"]],
        ],
        ignore_index=True,
    )
    combined = (
        combined.sort_values(series_cols + ["dt"])
        .drop_duplicates(subset=series_cols + ["dt"], keep="last")
        .reset_index(drop=True)
    )
    combined["token_delta"] = combined.groupby(series_cols, dropna=False)["token_position"].diff()
    combined["usdc_delta"] = combined.groupby(series_cols, dropna=False)["usdc_position"].diff()
    combined["token_delta"] = combined["token_delta"].fillna(combined["token_position"])
    combined["usdc_delta"] = combined["usdc_delta"].fillna(combined["usdc_position"])
    return combined

market_meta = (
    mdf[["condition_id", "question", "end_date_iso", "primary_tag"]]
    .drop_duplicates(subset=["condition_id"])
    .copy()
)
market_meta["end_date"] = pd.to_datetime(market_meta["end_date_iso"], utc=True, errors="coerce")
market_meta["market_close_dt"] = market_meta["end_date"].dt.floor("D") + pd.Timedelta(days=1)

plot_fills = fills.copy()
plot_fills["dt"] = pd.to_datetime(plot_fills["dt"], utc=True)
plot_fills = plot_fills.merge(market_meta, on="condition_id", how="left")
plot_fills["wallet_label"] = plot_fills["wallet"].map(_short_wallet)
plot_fills["market_label"] = plot_fills["question"].map(_short_text)
plot_fills["market_label"] = np.where(
    plot_fills["market_label"].eq("Unknown market"),
    plot_fills["condition_id"].str[:12],
    plot_fills["market_label"],
)

pnl_total_ts = _build_total_ts(plot_fills, "fill_pnl_usdc", "net_pnl_usdc", "cum_pnl_usdc")

wallet_pnl_totals = (
    plot_fills.groupby(["wallet", "wallet_label"], as_index=False, dropna=False)
    .agg(total_pnl_usdc=("fill_pnl_usdc", "sum"))
)
top_wallets_pnl = wallet_pnl_totals.reindex(
    wallet_pnl_totals["total_pnl_usdc"].abs().sort_values(ascending=False).index
).head(TOP_WALLETS_PNL).copy()
wallet_pnl_ts = (
    plot_fills[plot_fills["wallet"].isin(top_wallets_pnl["wallet"])]
    .groupby(["wallet", "wallet_label", "dt"], as_index=False, dropna=False)
    .agg(net_pnl_usdc=("fill_pnl_usdc", "sum"))
    .sort_values(["wallet", "dt"])
)

plot_fills["fill_pnl_usdc"] = pd.to_numeric(plot_fills["fill_pnl_usdc"], errors="coerce").fillna(0.0)
wallet_pnl_ts["cum_pnl_usdc"] = plot_fills["fill_pnl_usdc"]

market_pnl_totals = (
    plot_fills.groupby(["condition_id", "market_label", "end_date_iso"], as_index=False, dropna=False)
    .agg(total_pnl_usdc=("fill_pnl_usdc", "sum"))
)
top_markets_pnl = market_pnl_totals.reindex(
    market_pnl_totals["total_pnl_usdc"].abs().sort_values(ascending=False).index
).head(TOP_MARKETS_PNL).copy()
market_pnl_ts = (
    plot_fills[plot_fills["condition_id"].isin(top_markets_pnl["condition_id"])]
    .groupby(["condition_id", "market_label", "end_date_iso", "dt"], as_index=False, dropna=False)
    .agg(net_pnl_usdc=("fill_pnl_usdc", "sum"))
    .sort_values(["condition_id", "dt"])
)
market_pnl_ts["cum_pnl_usdc"] = market_pnl_ts.groupby("condition_id", dropna=False)["net_pnl_usdc"].cumsum()

granular_position_ts = _build_position_snapshots(plot_fills)

wallet_position_all = (
    granular_position_ts.groupby(["wallet", "wallet_label", "dt"], as_index=False, dropna=False)
    .agg(
        net_token_position=("token_delta", "sum"),
        net_usdc_position=("usdc_delta", "sum"),
    )
    .sort_values(["wallet", "dt"])
)
wallet_position_all["cum_token_position"] = wallet_position_all.groupby("wallet", dropna=False)["net_token_position"].cumsum()
wallet_position_all["cum_usdc_position"] = wallet_position_all.groupby("wallet", dropna=False)["net_usdc_position"].cumsum()
wallet_position_rank = (
    wallet_position_all.groupby(["wallet", "wallet_label"], as_index=False, dropna=False)
    .agg(max_abs_usdc_position=("cum_usdc_position", lambda s: s.abs().max()))
)
top_wallets_position = wallet_position_rank.sort_values("max_abs_usdc_position", ascending=False).head(TOP_WALLETS_POSITION).copy()
wallet_position_ts = wallet_position_all[wallet_position_all["wallet"].isin(top_wallets_position["wallet"])].copy()

market_position_all = (
    granular_position_ts.groupby(["condition_id", "market_label", "end_date_iso", "dt"], as_index=False, dropna=False)
    .agg(
        net_token_position=("token_delta", "sum"),
        net_usdc_position=("usdc_delta", "sum"),
    )
    .sort_values(["condition_id", "dt"])
)
market_position_all["cum_token_position"] = market_position_all.groupby("condition_id", dropna=False)["net_token_position"].cumsum()
market_position_all["cum_usdc_position"] = market_position_all.groupby("condition_id", dropna=False)["net_usdc_position"].cumsum()
market_position_rank = (
    market_position_all.groupby(["condition_id", "market_label", "end_date_iso"], as_index=False, dropna=False)
    .agg(max_abs_usdc_position=("cum_usdc_position", lambda s: s.abs().max()))
)
top_markets_position = market_position_rank.sort_values("max_abs_usdc_position", ascending=False).head(TOP_MARKETS_POSITION).copy()
market_position_ts = market_position_all[market_position_all["condition_id"].isin(top_markets_position["condition_id"])].copy()

token_total_ts = _build_total_ts(granular_position_ts, "token_delta", "net_token_position", "cum_token_position")
usdc_total_ts = _build_total_ts(granular_position_ts, "usdc_delta", "net_usdc_position", "cum_usdc_position")

print(
    f"Prepared plot datasets: total pnl={len(pnl_total_ts)}, "
    f"wallet pnl series={wallet_pnl_ts['wallet'].nunique()}, "
    f"market pnl series={market_pnl_ts['condition_id'].nunique()}, "
    f"wallet position series={wallet_position_ts['wallet'].nunique()}, "
    f"market position series={market_position_ts['condition_id'].nunique()}"
)


Prepared plot datasets: total pnl=1550, wallet pnl series=10, market pnl series=10, wallet position series=10, market position series=10


In [82]:
fig_pnl = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=(
        "Total cumulative PnL",
        f"Per-wallet cumulative PnL (top {TOP_WALLETS_PNL} by abs total PnL)",
        f"Per-market cumulative PnL (top {TOP_MARKETS_PNL} by abs total PnL)",
    ),
)

fig_pnl.add_trace(
    go.Scatter(
        x=pnl_total_ts["dt"],
        y=pnl_total_ts["cum_pnl_usdc"],
        mode="lines",
        name="Total",
        line=dict(width=3, color="#1f77b4"),
        hovertemplate="%{x|%Y-%m-%d %H:%M}<br>Total cum PnL: %{y:,.2f} USDC<extra></extra>",
    ),
    row=1,
    col=1,
)

wallet_totals_map = top_wallets_pnl.set_index("wallet")["total_pnl_usdc"].to_dict()
for wallet in top_wallets_pnl["wallet"]:
    sub = wallet_pnl_ts[wallet_pnl_ts["wallet"] == wallet]
    label = sub["wallet_label"].iloc[0]
    total = wallet_totals_map[wallet]
    fig_pnl.add_trace(
        go.Scatter(
            x=sub["dt"],
            y=sub["cum_pnl_usdc"],
            mode="lines",
            name=f"{label} ({total:,.1f})",
            hovertemplate=(
                f"Wallet: {wallet}<br>"
                + "%{x|%Y-%m-%d %H:%M}<br>"
                + "Cum PnL: %{y:,.2f} USDC<extra></extra>"
            ),
        ),
        row=2,
        col=1,
    )

market_totals_map = top_markets_pnl.set_index("condition_id")["total_pnl_usdc"].to_dict()
for condition_id in top_markets_pnl["condition_id"]:
    sub = market_pnl_ts[market_pnl_ts["condition_id"] == condition_id]
    label = sub["market_label"].iloc[0]
    end_date_iso = sub["end_date_iso"].iloc[0]
    total = market_totals_map[condition_id]
    fig_pnl.add_trace(
        go.Scatter(
            x=sub["dt"],
            y=sub["cum_pnl_usdc"],
            mode="lines",
            name=f"{label} ({total:,.1f})",
            hovertemplate=(
                f"Market: {label}<br>"
                + f"end_date_iso: {end_date_iso}<br>"
                + f"condition_id: {condition_id}<br>"
                + "%{x|%Y-%m-%d %H:%M}<br>"
                + "Cum PnL: %{y:,.2f} USDC<extra></extra>"
            ),
        ),
        row=3,
        col=1,
    )

for row in (1, 2, 3):
    fig_pnl.add_hline(y=0, line_dash="dash", line_color="grey", line_width=1, opacity=0.5, row=row, col=1)

fig_pnl.update_yaxes(title_text="USDC", row=1, col=1)
fig_pnl.update_yaxes(title_text="USDC", row=2, col=1)
fig_pnl.update_yaxes(title_text="USDC", row=3, col=1)
fig_pnl.update_xaxes(title_text="Time", row=3, col=1)
fig_pnl.update_layout(
    template="plotly_white",
    height=1200,
    title="Stage 2 copy-trade backtest - cumulative PnL",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
fig_pnl.show(renderer="browser")


In [83]:
fig_token_pos = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=(
        "Total signed token position",
        f"Per-wallet signed token position (top {TOP_WALLETS_POSITION} by max abs USDC position)",
        f"Per-market signed token position (top {TOP_MARKETS_POSITION} by max abs USDC position)",
    ),
)

fig_token_pos.add_trace(
    go.Scatter(
        x=token_total_ts["dt"],
        y=token_total_ts["cum_token_position"],
        mode="lines",
        line_shape="hv",
        name="Total token position",
        line=dict(width=3, color="#2ca02c"),
        hovertemplate="%{x|%Y-%m-%d %H:%M}<br>Signed token position: %{y:,.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)

for wallet in top_wallets_position["wallet"]:
    sub = wallet_position_ts[wallet_position_ts["wallet"] == wallet]
    label = sub["wallet_label"].iloc[0]
    max_abs = sub["cum_usdc_position"].abs().max()
    fig_token_pos.add_trace(
        go.Scatter(
            x=sub["dt"],
            y=sub["cum_token_position"],
            mode="lines",
            line_shape="hv",
            name=f"{label} ({max_abs:,.1f} USDC)",
            hovertemplate=(
                f"Wallet: {wallet}<br>"
                + "%{x|%Y-%m-%d %H:%M}<br>"
                + "Signed token position: %{y:,.2f}<extra></extra>"
            ),
        ),
        row=2,
        col=1,
    )

for condition_id in top_markets_position["condition_id"]:
    sub = market_position_ts[market_position_ts["condition_id"] == condition_id]
    label = sub["market_label"].iloc[0]
    end_date_iso = sub["end_date_iso"].iloc[0]
    max_abs = sub["cum_usdc_position"].abs().max()
    fig_token_pos.add_trace(
        go.Scatter(
            x=sub["dt"],
            y=sub["cum_token_position"],
            mode="lines",
            line_shape="hv",
            name=f"{label} ({max_abs:,.1f} USDC)",
            hovertemplate=(
                f"Market: {label}<br>"
                + f"end_date_iso: {end_date_iso}<br>"
                + f"condition_id: {condition_id}<br>"
                + "%{x|%Y-%m-%d %H:%M}<br>"
                + "Signed token position: %{y:,.2f}<extra></extra>"
            ),
        ),
        row=3,
        col=1,
    )

for row in (1, 2, 3):
    fig_token_pos.add_hline(y=0, line_dash="dash", line_color="grey", line_width=1, opacity=0.5, row=row, col=1)

fig_token_pos.update_yaxes(title_text="Tokens", row=1, col=1)
fig_token_pos.update_yaxes(title_text="Tokens", row=2, col=1)
fig_token_pos.update_yaxes(title_text="Tokens", row=3, col=1)
fig_token_pos.update_xaxes(title_text="Time", row=3, col=1)
fig_token_pos.update_layout(
    template="plotly_white",
    height=1200,
    title="Stage 2 copy-trade backtest - signed token positions",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
fig_token_pos.show(renderer="browser")


In [84]:
fig_usdc_pos = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=(
        "Total signed USDC position (using traded price)",
        f"Per-wallet signed USDC position (top {TOP_WALLETS_POSITION} by max abs USDC position)",
        f"Per-market signed USDC position (top {TOP_MARKETS_POSITION} by max abs USDC position)",
    ),
)

fig_usdc_pos.add_trace(
    go.Scatter(
        x=usdc_total_ts["dt"],
        y=usdc_total_ts["cum_usdc_position"],
        mode="lines",
        line_shape="hv",
        name="Total USDC position",
        line=dict(width=3, color="#d62728"),
        hovertemplate="%{x|%Y-%m-%d %H:%M}<br>Signed USDC position: %{y:,.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)

for wallet in top_wallets_position["wallet"]:
    sub = wallet_position_ts[wallet_position_ts["wallet"] == wallet]
    label = sub["wallet_label"].iloc[0]
    max_abs = sub["cum_usdc_position"].abs().max()
    fig_usdc_pos.add_trace(
        go.Scatter(
            x=sub["dt"],
            y=sub["cum_usdc_position"],
            mode="lines",
            line_shape="hv",
            name=f"{label} ({max_abs:,.1f})",
            hovertemplate=(
                f"Wallet: {wallet}<br>"
                + "%{x|%Y-%m-%d %H:%M}<br>"
                + "Signed USDC position: %{y:,.2f}<extra></extra>"
            ),
        ),
        row=2,
        col=1,
    )

for condition_id in top_markets_position["condition_id"]:
    sub = market_position_ts[market_position_ts["condition_id"] == condition_id]
    label = sub["market_label"].iloc[0]
    end_date_iso = sub["end_date_iso"].iloc[0]
    max_abs = sub["cum_usdc_position"].abs().max()
    fig_usdc_pos.add_trace(
        go.Scatter(
            x=sub["dt"],
            y=sub["cum_usdc_position"],
            mode="lines",
            line_shape="hv",
            name=f"{label} ({max_abs:,.1f})",
            hovertemplate=(
                f"Market: {label}<br>"
                + f"end_date_iso: {end_date_iso}<br>"
                + f"condition_id: {condition_id}<br>"
                + "%{x|%Y-%m-%d %H:%M}<br>"
                + "Signed USDC position: %{y:,.2f}<extra></extra>"
            ),
        ),
        row=3,
        col=1,
    )

for row in (1, 2, 3):
    fig_usdc_pos.add_hline(y=0, line_dash="dash", line_color="grey", line_width=1, opacity=0.5, row=row, col=1)

fig_usdc_pos.update_yaxes(title_text="USDC", row=1, col=1)
fig_usdc_pos.update_yaxes(title_text="USDC", row=2, col=1)
fig_usdc_pos.update_yaxes(title_text="USDC", row=3, col=1)
fig_usdc_pos.update_xaxes(title_text="Time", row=3, col=1)
fig_usdc_pos.update_layout(
    template="plotly_white",
    height=1200,
    title="Stage 2 copy-trade backtest - signed USDC positions",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
fig_usdc_pos.show(renderer="browser")
